# Review Text Analysis

This notebook analyzes skincare customer review text to identify recurring complaints, praise, and product improvement opportunities.

The goal is to move beyond ratings and review counts by understanding what customers actually say about products.

In [1]:
import pandas as pd

In [66]:
products = pd.read_csv(
    "../data/processed/products_clean.csv"
)

In [2]:
reviews = pd.read_csv(
    "../data/processed/skincare_reviews_clean.csv",
    parse_dates=["submission_time"]
)

/var/folders/gw/b8128pz11k31ky_873j0sxsc0000gn/T/ipykernel_49700/3526280538.py:1: DtypeWarning: Columns (0: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(


In [3]:
reviews.shape

(1092967, 18)

In [4]:
reviews[["rating", "review_text", "product_name", "brand_name"]].head()

,rating,review_text,product_name,brand_name
0,5,I use this with the Nudestix “Citrus Clean Bal...,Gentle Hydra-Gel Face Cleanser,NUDESTIX
1,1,I bought this lip mask after reading the revie...,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE
2,5,My review title says it all! I get so excited ...,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE
3,5,I’ve always loved this formula for a long time...,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE
4,5,"If you have dry cracked lips, this is a must h...",Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE


In [5]:
negative_reviews = reviews[reviews["rating"] <= 2].copy()
positive_reviews = reviews[reviews["rating"] >= 4].copy()

grouping the reviews based of neg and pos, 1-2 stars is negative, 4-5 is positive.

3 is neutral so leaving it out, from here can create clearer analysis

In [8]:
print("Negative reviews:", len(negative_reviews))
print("Positive reviews:", len(positive_reviews))

Negative reviews: 114061
Positive reviews: 897154


In [7]:
negative_reviews[
    ["rating", "product_name", "brand_name", "review_text"]
].head(10)

,rating,product_name,brand_name,review_text
1,1,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I bought this lip mask after reading the revie...
6,2,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I’ll give this 2 stars for nice packaging and ...
13,1,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Honestly I was so excited when I got this in t...
20,2,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,There’s nothing wrong with it but i think it w...
21,1,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Just like Aquaphor just get something cheaper...
24,2,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,Not worth the hype. My lips are still dry afte...
27,1,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,i really tried to love this ☹️ i wore it to sl...
30,1,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,dried my lips out sooooooo bad. literally were...
31,1,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,"I used this product daily last week, and my li..."
32,2,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I really loved this product and did use the wh...


In [9]:
positive_reviews[
    ["rating", "product_name", "brand_name", "review_text"]
].head(10)

,rating,product_name,brand_name,review_text
0,5,Gentle Hydra-Gel Face Cleanser,NUDESTIX,I use this with the Nudestix “Citrus Clean Bal...
2,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,My review title says it all! I get so excited ...
3,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I’ve always loved this formula for a long time...
4,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,"If you have dry cracked lips, this is a must h..."
5,4,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,The scent isn’t my favourite but it works grea...
7,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I use this at night or while I’m putting makeu...
8,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I love this stuff. I first had the sample size...
9,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I purchased the Sweet Candy scent at my local ...
10,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,this product is a bit pricey but after using o...
11,5,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,I use this every night and morning and it work...


this rating-based grouping provides an interpretable baseline before applying text-based sentiment or theme detection

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

In [12]:
negative_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 3),
    min_df=50,
    max_features=5000
)

In [13]:
negative_matrix = negative_vectorizer.fit_transform(
    negative_reviews["review_text"]
)

In [14]:
negative_phrase_counts = negative_matrix.sum(axis=0).A1

negative_phrases = pd.DataFrame({
    "phrase": negative_vectorizer.get_feature_names_out(),
    "count": negative_phrase_counts
})

negative_phrases = negative_phrases.sort_values(
    "count",
    ascending=False
)

negative_phrases.head(30)

,phrase,count
3588,sensitive skin,8327
1070,dry skin,7122
4757,wanted love,5840
3324,really wanted,4643
2760,oily skin,4490
1355,feel like,4218
4656,using product,3514
48,acne prone,3332
1139,excited try,3286
1437,felt like,3180


In [15]:
complaint_themes = {
    "dryness": [
        "dry", "drying", "dried", "dehydrated"
    ],
    "irritation": [
        "irritated", "irritation", "burning", "burned", "sting", "stinging"
    ],
    "breakouts": [
        "breakout", "breakouts", "broke out", "acne", "pimples"
    ],
    "ineffective": [
        "didn work", "doesn work", "no difference", "not effective"
    ],
    "value": [
        "waste money", "not worth", "too expensive", "overpriced"
    ],
    "texture": [
        "sticky", "greasy", "oily", "heavy", "thick"
    ],
    "smell": [
        "smell", "scent", "fragrance"
    ],
    "packaging": [
        "pump", "bottle", "packaging", "container", "leak"
    ]
}

In [16]:
def count_theme_mentions(df, text_column, themes):
    results = []

    text = df[text_column].str.lower()

    for theme, keywords in themes.items():
        pattern = "|".join(keywords)

        mentions = text.str.contains(
            pattern,
            regex=True,
            na=False
        )

        count = mentions.sum()
        percent = (count / len(df)) * 100

        results.append({
            "theme": theme,
            "review_count": count,
            "percent_of_reviews": round(percent, 2)
        })

    return pd.DataFrame(results).sort_values(
        "percent_of_reviews",
        ascending=False
    )

In [17]:
negative_theme_results = count_theme_mentions(
    negative_reviews,
    "review_text",
    complaint_themes
)

negative_theme_results

,theme,review_count,percent_of_reviews
6,smell,24666,21.63
5,texture,24255,21.26
0,dryness,24196,21.21
2,breakouts,19606,17.19
7,packaging,11585,10.16
1,irritation,9122,8.00
4,value,6005,5.26
3,ineffective,1992,1.75


In [18]:
complaint_themes = {
    "dryness": [
        "too dry",
        "too drying",
        "dried my skin",
        "dried out",
        "made my skin dry",
        "left my skin dry",
        "skin felt dry"
    ],

    "irritation": [
        "irritated my skin",
        "caused irritation",
        "burned my skin",
        "burning sensation",
        "made my skin burn",
        "stung my skin",
        "caused redness"
    ],

    "breakouts": [
        "broke me out",
        "broke out",
        "caused breakouts",
        "caused acne",
        "gave me acne",
        "caused pimples"
    ],

    "ineffective": [
        "didn't work",
        "did not work",
        "doesn't work",
        "does not work",
        "no difference",
        "saw no difference",
        "not effective",
        "did nothing"
    ],

    "value": [
        "waste of money",
        "not worth",
        "too expensive",
        "overpriced",
        "not worth the price"
    ],

    "texture": [
        "too sticky",
        "felt sticky",
        "too greasy",
        "felt greasy",
        "too heavy",
        "felt heavy",
        "too thick"
    ],

    "smell": [
        "smells bad",
        "bad smell",
        "strong smell",
        "strong scent",
        "too fragranced",
        "fragrance too strong",
        "hate the smell",
        "hate the scent"
    ],

    "packaging": [
        "pump broke",
        "bottle broke",
        "packaging broke",
        "leaking bottle",
        "leaked everywhere",
        "hard to dispense",
        "packaging is terrible"
    ]
}

In [19]:
dryness_mask = negative_reviews["review_text"].str.lower().str.contains(
    "too dry|too drying|dried my skin|dried out|made my skin dry|left my skin dry|skin felt dry",
    regex=True,
    na=False
)

negative_reviews.loc[
    dryness_mask,
    ["rating", "product_name", "review_text"]
].sample(10, random_state=42)

,rating,product_name,review_text
296369,1,Facial Radiance Pads,The girl at Sephora reccomended this one for m...
55694,2,Green Clean Makeup Removing Cleansing Balm,"I like that it removes everything smoothly, bu..."
519084,1,Find Your Balance Oil Control Cleanser,"I have oily skin with acne prone skin, and NOP..."
619735,2,Acne Solutions Clinical Clearing Gel,This works very well in eliminating that “bump...
245816,2,Oat Cleansing Balm,"It works well to remove makeup, but is too dry..."
219023,2,Mini Unseen Sunscreen SPF 40 PA+++,Giving it two stars because it feels heavy on ...
716056,1,Umbra Tinte Physical Daily Defense SPF 30,This product worked terribly on me. I have oil...
287891,1,Beste No. 9 Jelly Cleanser,I got this from last year’s birthday gift. I’v...
72578,2,Protini Polypeptide Firming Refillable Moistur...,Wanted to love this! Totally dried my skin out...
142694,1,Jet Lag Mask,"I have very normal skin, not too oily and not ..."


In [20]:
pd.set_option("display.max_colwidth", None)

In [21]:
negative_reviews.loc[
    dryness_mask,
    ["rating", "product_name", "review_text"]
].sample(10, random_state=42)

,rating,product_name,review_text
296369,1,Facial Radiance Pads,"The girl at Sephora reccomended this one for me as, she said it would be suitable for my sensitive skin. WRONG, it dried out my skin and I had some sort of reaction which left my face bumpy in a hive like manner. It’s been over a week now and my face is still rough and bumpy even with using a moisturizer. Would not reccomend if you have even a bit of skin sensitivity"
55694,2,Green Clean Makeup Removing Cleansing Balm,"I like that it removes everything smoothly, but it has made my face way too dry. Also it has an unpleasant scent, like a lemon scented floor cleaner. Probably better for oily skin types."
519084,1,Find Your Balance Oil Control Cleanser,"I have oily skin with acne prone skin, and NOPE!! This was not it for me! My skin dried out completely & now I have new breakouts :(."
619735,2,Acne Solutions Clinical Clearing Gel,This works very well in eliminating that “bump“ when you feel it. but then it dried out my skin badly and actually made it hard . It also burns when you apply it.
245816,2,Oat Cleansing Balm,"It works well to remove makeup, but is too drying for my combination skin. I have dry cheeks and an oily t-zone."
219023,2,Mini Unseen Sunscreen SPF 40 PA+++,Giving it two stars because it feels heavy on my face and suffocating to my skin. I put it on and as others have said it doesn’t soak in. It doesn’t feel like other primers that I have used with SPF. I can still feel it on my face. I got it because my skin is dry/oily. My T zone gets oily while my cheeks and neck get very dry. I tried Elta 46 for face but it dried out my cheeks and neck and I had very visible dry patches. I read it was because of the zinc oxide. So now I’m back to square one trying to find a face sunscreen that works well under makeup and works well with my combo skin and doesn’t feel like I have a face mask on. I’ve tried Cerave AM and my makeup just slid right off.
716056,1,Umbra Tinte Physical Daily Defense SPF 30,This product worked terribly on me. I have oily skin and was recommended this by a Sephora employee. It dried out my skin and caused my skin to purge. I’m very unhappy with how the product performed especially since I deal with acne and I finally got to a point where my skin was clear.
287891,1,Beste No. 9 Jelly Cleanser,"I got this from last year’s birthday gift. I’ve had positive experiences with other Drunk Elephant products before, so I had good expectations with this cleanser. For reference, I have oily, non sensitive skin. This cleanser broke me out on my entire face, both small pimples and painful cystic acne. It also dried out my skin and made it sensitive to the other products I use. Every time I’d put on a serum and moisturizer, it would burn my skin. I ended up switching to a cetaphil cleanser I had and that worked so much better than this one. My skin was finally clearing up before using this cleanser, and now I have to deal with the acne scars caused by this. Do not get this!!"
72578,2,Protini Polypeptide Firming Refillable Moisturizer,Wanted to love this! Totally dried my skin out- felt oily and tat after three days of use. Little angry red bumps appeared after a week- love drunk elephant but this didn’t work for me.
142694,1,Jet Lag Mask,"I have very normal skin, not too oily and not too dry. Heard great things about and decided to give it a try, bought it, used it 3 times and it broke me out so bad. Plus doesn’t smell too good. Product didn’t work out for me."


In [22]:
irritation_mask = negative_reviews["review_text"].str.lower().str.contains(
    "irritated my skin|caused irritation|burned my skin|burning sensation|made my skin burn|stung my skin|caused redness",
    regex=True,
    na=False
)

negative_reviews.loc[
    irritation_mask,
    ["rating", "product_name", "review_text"]
].sample(10, random_state=42)

,rating,product_name,review_text
736907,1,GLO Brilliant White Smile - At Home Teeth Whitening Device,"I was really looking forward to trying this new system, but I’m returning it only after 2 tries. I can’t vouch for whether it will actually whiten your teeth or not.Even during my first try, I had an uncomfortable, burning sensation from the gel once it touched my lips. I didn’t have any sensitivity regarding my teeth, but because the gel packaging is difficult to handle, the gel ususally got on my lips, which wouldn’t bother me if it didn’t cause such a horrible burning feeling. Maybe it is something in the ingredients that I am sensitive to, but I couldn’t bear using it again after the first two tries. So, I’ll be returning this product."
916730,1,Powerful-Strength Vitamin C Serum,Didn’t like burning sensation - skin turned red
266897,1,Mini Facial Treatment Essence (Pitera Essence),"Seriously this messed up my skin so freaking badly. I got this with the face lotion in the rewards bazar and the these ingredients irritated my skin so badly. I have these red bumps all over my face and it If I try to soothe my skin with anything besides coconut oil it burns. My skin was in so much better condition before using this and I completely regret using this. I only used the sample for 3 days and then realized my negative skin reaction was from this and not the other product I started around the same time or the sample lotion. I’m so disappointed and it’s been 4 days since I stopped using it and my skin looks so bad it’s embarrassing. I would not recommend purchasing this unless you do a sample test in store since it is a pricey product. I would post a picture of the before and after of my skin but the last time I tried to do that with the negative side effects my skin suffered from one of the glam glow masks, Sephora wouldn’t post my review."
575798,2,Mini Precleanse Cleansing Oil,"I don’t understand how this product has so much hype. The fragrance was so strong, and it irritated my skin within 20 seconds of it being on my skin. I gave it 2 stars because it DOES remove makeup pretty well, but this is not the product for sensitive skin."
229700,1,EradiKate Acne Treatment,"I don’t have sensitive skin and applied this carefully as directed on my budding acne. It did not work to reduce, prevent, minimize or help with the acne and burned my skin horribly, leaving a mark that took weeks to fade. This is the first thing I ever returned to Sephora. I was really disappointed because so many other people seemed to love it. If you try it, use sparingly and do some skin tests first."
906373,1,Acid Potion AHA + BHA Resurfacing Exfoliator,This product is way over priced. Found it to be ineffective and completely irritated my skin and my skin isn’t particularly sensitive either. I wouldn’t recommend this.
759811,2,Aqua Bomb Jelly Cleanser,"First few uses, it felt lathered up really nice and smelled very pleasant. After a few more uses, I found that it would leave my face stinging and give it a raw-feeling when I apply the rest of my skincare. After discontinuing the use of the product, my skin went back to normal. I suspect it’s either the heavy fragrance additives or a harsh surfactant that irritated my skin. Definitely don’t recommend for people with sensitive skin."
276989,1,Self Tanning Bronzing Face Drops,"Super streaky and I followed the instructions! But again if your mixing in white lotion there’s really no way to know if it’s rubbed in well. The color definitely showed but it was streaky and after trying it on day 4, the solution irritated my skin."
763396,1,The POREfessional Tight ’n Toned Pore-Refining AHA+PHA Toner,"In the past I have really liked every makeup product I’ve gotten from Benefit. I was surprised and excited to see they were expanding into skincare. However, I am not a fan of this toner. The toner did not feel gentle and even slightly burned my skin when I applied it on clean skin. The packaging and the fact that it is a foam

In [ ]:
#we are validating that the algorithm actually works in the way it should for each theme

In [23]:
breakouts_mask = negative_reviews["review_text"].str.lower().str.contains(
    "broke me out|broke out|caused breakouts|caused acne|gave me acne|caused pimples",
    regex=True,
    na=False
)

negative_reviews.loc[
    breakouts_mask,
    ["rating", "product_name", "review_text"]
].sample(10, random_state=42)

,rating,product_name,review_text
358891,2,U.F.O. Salicylic Acid BHA Acne Treatment Face Oil,"smells like herbs, but not in a good way - more like a kitchen. it also broke me out and my skin never cleared up. still used the product because they say it could just be “purging“. nope, just not a good product for me. i think i got i’ll from the scent"
352473,2,Superberry Hydrate + Glow Dream Night Mask with Vitamin C,"My face said it’s a no, broke out so bad from it. It is very moisturizing though"
213833,2,Checks and Balances Frothy Face Wash,I’ve always struggled with acne. I have slightly oily skin but nothing major. I started using this and noticed it made me quite oily and I think it broke me out some. I did like how much it lathered up. I switched to their Zero Oil face wash instead.
764695,1,15% Vitamin C and EGF Brightening Serum,After spending so much time to improve my skin. I have to start all over this stuff broke me out so bad. My skin became red and inflamed with patchy dry spots with just 5 days of use. I has to throw it out.
867343,2,Nourishing Moisturizer with Prebiotics,"The previous formula of the Nourishing Moisturizer was so much better - it was the only moisturizer I used for years, it made my skin feel amazing and never broke me out. This new version with “prebiotics“ leaves my skin feeling dry and tight. :( Will not repurchase."
807203,1,Collagen Booster Firming Peptide Serum,Broke me out and there is barley any product. Not buying again.
114584,1,Pure Skin Face Cleanser,"I heard so many great things about this cleanser for people with skin sensitivity, however THIS BROKE ME OUT SO BAD. I have dry skin with keratosis pilaris and am easily prone to redness. This is marketed as a gentle cleanser, so I thought it would be fine. Let me tell you, this stuff caused the worst break out of my life!! Honestly I was shocked that one product could completely destroy my skin like this one did. I immediately switched to philosophy Purity and have had amazing success. If you really want to try this get the travel size and see how your skin reacts, just be warned that this may not be so great for people with sensitive skin."
825069,1,Ultra Repair Firming Collagen Cream with Peptides and Niacinamide,"Broke out after one use. I’m hunting for cruelty free moisturizers for mature skin and the reviews seemed to point to this jar. Sadly, it’s on its way back."
216969,2,Unseen Sunscreen SPF 40 PA+++,Love the texture and the finish on my face - it feels super lightweight and I can’t even tell I have it on. 2 stars because it broke me out like crazy (I have combo acne prone skin) and didn’t do much to protect my face from the sun.
781681,2,Pink Cloud Rosewater + Squalane Makeup Removing Face Wash,"I was excited to try this cleanser since the packaging is beautiful and the price point is reasonable, but this is really just a cheap cleanser dressed up to look pretty. It smells and feels like Cetaphil that’s been on the shelf a little too long, and it broke me out in little bumps. I do not recommend."


In [25]:
#now the word "ineffective"

ineffective_mask = negative_reviews["review_text"].str.lower().str.contains(
    "didn't work|did not work|doesn't work|does not work|no difference|saw no difference|not effective|did nothing",
    regex=True,
    na=False
)

negative_reviews.loc[
    ineffective_mask,
    ["rating", "product_name", "review_text"]
].sample(10, random_state=42)

,rating,product_name,review_text
482951,1,Mini Anti-Aging Cleansing Gel,Used exactly as directed and it burned and dried out my oily-combo skin terribly. Backed down to using it twice a week and still had issues. Did nothing for my fine lines and did not improve any element of my complexion. Sent it back.
262030,1,Facial Treatment Essence (Pitera Essence),"I was so excited to try this product but I just don’t get the hype. Maybe it’s because I only received a deluxe sample and it just didn’t have enough time to work it magic. I found it difficult to apply. The directions say to splash in your hands and pat on which I tried but I felt like that wasted product by running too easily. If using a cotton ball, too much absorbs and then even more product is wasted. It made my skin feel dry and tight which is ironic since it’s supposed to be hydrating. In the end I saw no difference in my skin at all after using it."
684973,1,Rose Floral Toner,"I have been using this product for over a month, every morning and I don’t see any difference in my skin. When I was in a Sephora store the worker told me it would help with the redness on my face but I see no difference. It smells amazing but not worth the price."
559114,1,Lactic Acid 10% + HA 2% Exfoliating Serum,"I was so excited to try this when I bought it. Unfortunately, it does not work at all. For the first 2 weeks, I diluted this with hyaluronic acid because this was my first time using an acid and I was afraid that I would get some terrible reaction. Some people noticed a difference after using even a little bit, but I didn’t.A couple weeks ago, I started using it again every night (NOT diluted this time), and still, I didn’t notice any difference. It does make my face seem a bit sticky, but that’s not really a big concern since you’re using it at nice.I’m really disappointed it didn’t work :("
554908,2,Cooling Water,I saw no difference using this whatsoever. I resolved to using this on my neck until it runs out.
616963,1,Pure Argan Milk Intensive Hydrating Treatment,"I don’t usually review products I got as a sample but I have very dry patches on my face and wanted toTry something extra hydrating. The product smelled awful, which isn’t super important to me usually but it didn’t really go away for a long time. It sat on top of my skin and did nothing for my patches. And even after washing my face well that night I woke up with more acne than I’ve seen on my face in years. If you have sensitive skin maybe stay away from this."
845175,2,Teatreement Cleansing Foam,"IT DID NOT FOAM AT ALL. Maybe I received a defective sample? And like the other products in this line, it has an overwhelming and unpleasant, almost medicinal scent. It did nothing for my hormonal acne, but might be more suited for other types of acne. However, it is great at removing makeup! Completely removed heavy foundation without using a prior makeup remover. That being said though, I would still not recommend this due to not living up to its claims and its unpleasant scent."
257740,2,Sugar Advanced Lip Balm Intense Hydration Treatment,Had high hopes for this product after reading reviews but this did nothing for my lips and it’s not even winter yet. I’ll stick to Rosebud which is very effective and a fraction of the price.
300264,1,B-Hydra Intensive Hydration Serum with Hyaluronic Acid,This doesn’t provide enough hydration. My skin feels tight after using it. I apply twice because once is not enough and I still go over it with argan oil to not feel dry. My skin type is combination and this does not work for me.
1082524,1,Mini Squalane + Vitamin C Rose Firming Oil,"It did nothing I really don’t get the excitement ., the bottle is Cute enough about it."


In [26]:
value_mask = negative_reviews["review_text"].str.lower().str.contains(
    "waste of money|not worth|too expensive|overpriced|not worth the price",
    regex=True,
    na=False
)

negative_reviews.loc[
    value_mask,
    ["rating", "product_name", "review_text"]
].sample(10, random_state=42)

,rating,product_name,review_text
198470,2,T.L.C. Framboos Glycolic Resurfacing Night Serum,Just not worth the price. Didn’t notice any difference :(
444349,2,Moisture Surge 100H Auto-Replenishing Hydrator Moisturizer,"I have dry skin.Before applying, my skin felt and looked dry and dull. After applying, my skin felt smooth and less dull but under the surface the skin was still dry. The effect lasts all day or until you wash your face.It’s a very light weight gel moisturizer. A little goes a long way. I would recommend using this on top of an actual hydrating moisturizer. I would not buy this. It’s not worth this price."
299236,1,B-Hydra Intensive Hydration Serum with Hyaluronic Acid,This product was not a good serum for moisturizing after applying once about 30 mins later I can See dry flakes on my face and after reapplying same thing again and again I have Very dry skin and would not recommend this product. Not worth the money either
7012,2,Lip Sleeping Mask Intense Hydration with Vitamin C,"Overhyped! This product smells great , but I wouldn’t call it a “mask” it does absolutely nothing for your lips. You won’t wake up with “super soft lips” like everyone claims. Vaseline is only 3 bucks and does a much better job than this overpriced Vaseline. Won’t repurchase."
220349,2,Mini Unseen Sunscreen SPF 40 PA+++,Nothing spectacular in my opinion and overpriced. I don’t really care for the consistency of it and although I haven’t burned I think I am getting tan on my face which I hate! But I’m going to use it since it was so expensive. I like the fragrance in it but if your sensitive to that do not buy this.
70474,1,Protini Polypeptide Firming Refillable Moisturizer,Not worth the hype..Dried my already dry skin out more.IMO this entire line is overrated.
266506,1,Mini Facial Treatment Essence (Pitera Essence),"I really wanted to love this after getting targeted ads because of my acne prone skin, and reading all these great reviews. I initially got a deluxe sample, so I figured I’d give it a shot. I’ve been using it for 5 months now day and night, and have noticed 0 change in my skin whatsoever. Wayyy too expensive to not seeing any difference :("
700286,1,Mini Faded Serum for Dark Spots & Discoloration,Used almost an entire tube with 0 improvements of dark spots. Stinks to the high heavens (yes it is that bad). Packaging is also not good because the serum often leaks out of the tip and my tube also has a hole in it after regular use so all the remaining serum leaked out! I really wanted to love this one but I definitely won’t be repurchasing. Total waste of money. :((
303015,2,T.L.C. Sukari Babyfacial AHA + BHA Mask,This product is not worth the price. I could get the same results with less expensive products.. Results were not instant. This product is a bust.
1078677,2,Hyaluronic Acid + Peptide Lip Treatment Booster,"I wanted to like this, but it’s not worth the price at all. The metal applicator seems good, but it makes the actual balm taste like metal on your lips (despite the contents being flavorless). It hydrates, but it doesn’t last. For the price and the fact you have to keep applying, I’d suggest other options."


In [ ]:
# validation record is all

# dryness:      10/10
# irritation:   10/10
# breakouts:    10/10
# ineffective:  10/10
# value:        10/10

In [27]:
texture_mask = negative_reviews["review_text"].str.lower().str.contains(
    "too sticky|felt sticky|too greasy|felt greasy|too heavy|felt heavy|too thick",
    regex=True,
    na=False
)

negative_reviews.loc[
    texture_mask,
    ["rating", "product_name", "review_text"]
].sample(5, random_state=42)

,rating,product_name,review_text
340609,2,Mini Advanced Night Repair Synchronized Multi-Recovery Complex,"I received this product complimentary from Influenster for testing purposes. All reviews are my own and unbiased. The product I received was a sample size containing approximately 7 days worth of product, so this must be taken into account. The texture of this serum is a nice texture. Not too thick or too runny. The scent is light and unbothersome. After a week of use, I did notice my skin was more hydrated, but other than that, there was no difference. I did not notice any improvement in fine lines and wrinkles, nor did I notice improvement in radiance. I would not recommend this product. There are better products on the market for this purpose. I am a 39 yrs old Female with fine lines around my eyes and on forehead."
480196,1,Multi-Peptide + HA Serum,"Wanted to love this but it feels far too sticky on my face alone or under makeup, feels like it never really dries OR absorbs, just sits on my face being sticky. Can’t meaningfully say if it does its job given I really just can’t stand how it feels on."
486726,1,Advanced Génifique Radiance Boosting Face Serum,"I tried a 7 day sample of Lancome Advanced Genifique Youth Activating Serum. When I try a new product, I don’t try anything else during that time period so that I can attribute any changes to the new product. In addition, I also evaluate what happens after I stop using it.The samples come in pre-measured packets - one packet for each day. The large amount of serum in each packet made me think, “I’m supposed to use this much every night?!?“ I slathered the whole amount on my face and neck and felt sticky. I felt like I had a mask on all night and itched because of it. If this excessive amount is what Lancome recommends be applied on a nightly basis, they must make a huge profit on this serum from re-orders.I didn’t see any changes in my skin until the last day, when I woke up with several pimples (whiteheads) across my face. There was no improvement in my fine lines, wrinkles, skin texture, radiance, etc. I’m nearing 50 years old, so these were the benefits that I was looking for. The only improvement was a couple of days after I finished the using the serum when my faced cleared up.Research regarding the main ingredient (Bifida Ferment Lysate - a type of yeast) is limited and says that it may provide some protection against oxidative damage in the skin. However, there are many other well-researched (and cheaper) ingredients that are proven to effectively combat oxidative damage (like vitamin C). I’m going to spend my money on those other products."
963916,2,"Vitamin B, C, and E Moisturizer",its okay but if your skin needs true moisture this ain’t it. my face felt greasy and dry at the same time
797680,2,Ultimate Sun Protector Cream SPF 50+ Face Sunscreen,"I usually love products from this brand so I was excited to try this sunscreen. The sunscreen is very difficult to rub in and leaves a white film on the face even after a few minutes and attempting to rub it in. I wore this product while running and it did seem to be water and sweat resistant. I prefer an SPF that would go well under makeup and is weightless, this one is a little too thick to wear under my makeup."


In [28]:
smell_mask = negative_reviews["review_text"].str.lower().str.contains(
    "smells bad|bad smell|strong smell|strong scent|too fragranced|fragrance too strong|hate the smell|hate the scent",
    regex=True,
    na=False
)

negative_reviews.loc[
    smell_mask,
    ["rating", "product_name", "review_text"]
].sample(5, random_state=42)

,rating,product_name,review_text
684547,2,Rose Floral Toner,"I am astonished there are not more reviews about the strong smell (maybe I’m missing them?) In any case, I knew by a single whiff that this was not for me. IMO any product with rose in it is hit or miss because it is one of those scents that can easily come off as overpowering. One of the possible culprits in this that makes it too strong smelling is the addition of chamomile flower extract ie. Anthemis nobilis. The advertised strengths of this toner would be its cooling sensation and gentleness which sounds appealing. However I couldn’t bring myself to spray on my face. If you are leery of heavy scents, I strongly suggest sampling on a cotton ball or something other than your skin. I received this in Play!"
463120,1,Superfood Air-Whip Lightweight Moisturizer with Hyaluronic Acid,"The consistency and spread are terrible. It feels cheap and tough. This is not a good product and the reviews love it. Like, did I get a bad batch or something? I finally threw it out after a few tries because it’s not even good enough to be a backup purse moisturizer. Also, it smells bad, like soft cucumbers."
485063,1,C-Firma Fresh Vitamin-C Day Serum,"I was so excited to try this, but it truly did not work out for me at all. First, I had an extreme reaction to it. If you have sensitive skin and/or are not used to vitamin C, I would recommend avoiding this and opting for a more gentle one. My face was burned for days after using it. Secondly, while I can appreciate that you don’t mix the powder and liquid until you get it so that the active ingredients stay fresh longer, it was a MESS to mix. Lastly, this has a strong scent, and it doesn’t fade away during the day. I feel bad, but I can’t find a single positive thing to say about this. It is so extremely expensive, but I don’t think I’d buy it even if it would be $10."
945721,2,Honey Whip Peptide and Collagen Moisturizer,"I can’t get past the bad smell. It smells like the cleaner used in gas station bathrooms. Not very pleasant. It does absorb quickly, but the smell lingers for a while."
382195,1,Squalane + Omega Repair Deep Hydration Moisturizer,I got this as a sample. I didn’t notice any bad smell as others have stated. I don’t have sensitive skin by any means. I started to notice a bunch of little red bumps around my cheeks and down to my chin. I finally stopped using this product and it took about a month for my skin to go back to normal


In [29]:
packaging_mask = negative_reviews["review_text"].str.lower().str.contains(
    "pump broke|bottle broke|packaging broke|leaking bottle|leaked everywhere|hard to dispense|packaging is terrible",
    regex=True,
    na=False
)

negative_reviews.loc[
    packaging_mask,
    ["rating", "product_name", "review_text"]
].sample(5, random_state=42)

,rating,product_name,review_text
965809,1,Cream Skin Mist,"The product itself is fine and I like the fine mister, but the packaging is terrible. I have used it less than a dozen times, but I only have 30% of the bottle left. Every time I spray, it leaks heavily. One time it sat on my counter—I kid you not—and at least half an ounce just dripped out onto my counter. The next time I went to use it, there was 1/2 an inch of liquid sitting inside the cap, which of course poured out when I opened it. No clue how it even happened. Very frustrating, and I’m afraid to purchase again because of how much was wasted."
480903,2,Anti-Aging Cleansing Gel,This face wash left my skin feeling so dry and tight after using it every night for about a week. It also has a pretty strong fragrance and the packaging is terrible. I brought this with me traveling where it spilled all over my bag and ruined the box that it came in (even though the cap was screwed on all of the way).
1033621,2,FAB Skin Lab Retinol Serum 0.25% Pure Concentrate,"Product was fine, didn’t cause breakouts and didn’t sensitize my skin when used appropriately. Results were average. The pump on the bottle broke and product came out the sides so I returned it...only to have the new one do the same thing. Better products and better packaging out there."
300597,1,T.L.C. Sukari Babyfacial AHA + BHA Mask,Pump broken or not as much product as advertised on bottle. Only was able to use twice a very small amount.
913410,1,"Glow Clear, Color Correcting Self-Tanning Mousse",Don’t waste your money. Pump broke within 4 pumps of using the product and then continued to spill out of the top. Very orange as well for claiming to not have an orange undertone.


In [ ]:
# texture:    4/5 valid flagged as issue but wasnt
# smell:      4/5 valid
# packaging:  5/5 valid

In [33]:
complaint_themes = {
    "dryness": [
        "too dry",
        "too drying",
        "dried my skin",
        "dried out",
        "made my skin dry",
        "left my skin dry",
        "skin felt dry"
    ],

    "irritation": [
        "irritated my skin",
        "caused irritation",
        "burned my skin",
        "burning sensation",
        "made my skin burn",
        "stung my skin",
        "caused redness"
    ],

    "breakouts": [
        "broke me out",
        "broke out",
        "caused breakouts",
        "caused acne",
        "gave me acne",
        "caused pimples"
    ],

    "ineffective": [
        "didn't work",
        "did not work",
        "doesn't work",
        "does not work",
        "no difference",
        "saw no difference",
        "not effective",
        "did nothing"
    ],

    "value": [
        "waste of money",
        "not worth",
        "too expensive",
        "overpriced",
        "not worth the price"
    ],

    "texture": [
        "too sticky",
        "felt sticky",
        "too greasy",
        "felt greasy",
        "too heavy",
        "felt heavy",
        "way too thick",
        "very thick",
        "too thick for"
    ],

    "smell": [
        "smells bad",
        "smell is bad",
        "strong smell",
        "strong scent",
        "too fragranced",
        "fragrance too strong",
        "hate the smell",
        "hate the scent",
        "unpleasant smell",
        "unpleasant scent"
    ],

    "packaging": [
        "pump broke",
        "bottle broke",
        "packaging broke",
        "leaking bottle",
        "leaked everywhere",
        "hard to dispense",
        "packaging is terrible"
    ]
}

In [34]:
negative_theme_results = count_theme_mentions(
    negative_reviews,
    "review_text",
    complaint_themes
)

negative_theme_results

,theme,review_count,percent_of_reviews
4,value,7617,6.68
3,ineffective,7327,6.42
2,breakouts,6588,5.78
0,dryness,2654,2.33
5,texture,2187,1.92
1,irritation,1346,1.18
6,smell,987,0.87
7,packaging,147,0.13


In [35]:
praise_themes = {
    "hydration": [
        "very hydrating",
        "so hydrating",
        "keeps my skin hydrated",
        "skin feels hydrated",
        "made my skin soft",
        "left my skin soft",
        "moisturizing",
        "very moisturizing"
    ],

    "effective": [
        "really works",
        "works great",
        "worked for me",
        "saw a difference",
        "noticed a difference",
        "great results",
        "made a difference",
        "actually works"
    ],

    "gentle": [
        "very gentle",
        "gentle on my skin",
        "didn't irritate",
        "did not irritate",
        "good for sensitive skin",
        "no irritation",
        "doesn't sting",
        "does not sting"
    ],

    "texture": [
        "lightweight",
        "absorbs quickly",
        "absorbed quickly",
        "not greasy",
        "not sticky",
        "smooth texture",
        "feels smooth",
        "silky texture"
    ],

    "glow": [
        "glowing skin",
        "skin is glowing",
        "made my skin glow",
        "brighter skin",
        "skin looks brighter",
        "radiant",
        "more radiant"
    ],

    "value": [
        "worth the money",
        "worth the price",
        "great value",
        "good value",
        "lasts a long time"
    ],

    "packaging": [
        "love the packaging",
        "great packaging",
        "easy to use",
        "easy to dispense",
        "love the pump"
    ]
}

In [36]:
positive_theme_results = count_theme_mentions(
    positive_reviews,
    "review_text",
    praise_themes
)

positive_theme_results

,theme,review_count,percent_of_reviews
3,texture,62828,7.00
0,hydration,56628,6.31
1,effective,31121,3.47
4,glow,19815,2.21
5,value,16108,1.80
2,gentle,15708,1.75
6,packaging,14817,1.65


In [37]:
texture_positive_mask = positive_reviews["review_text"].str.lower().str.contains(
    "lightweight|absorbs quickly|absorbed quickly|not greasy|not sticky|smooth texture|feels smooth|silky texture",
    regex=True,
    na=False
)

positive_reviews.loc[
    texture_positive_mask,
    ["rating", "product_name", "review_text"]
].sample(5, random_state=42)

,rating,product_name,review_text
927903,5,Pep-Start Daily UV Protector Broad Spectrum SPF 50,"I live in AZ, so I need sunscreen every day. For years and years, the only sunscreen product that hasn’t irritated my face or eyes, caused breakouts, and stayed matte on my skin was Clinique City Block Sheer SPF 25. Even though I tan easily, I want to avoid melasma, and other damage caused by the sun. Recently I’ve been told by my dermatologist that SPF 25 isn’t enough coverage for incidental exposure, in this sunny climate. She recommends SPF 30 for every day (!), and SPF 50 for prolonged time in the sun. I thought I’d been doing well with City Block SPF 25 and Guerlain Lingerie de Peau foundation SPF 20, but, apparently, that’s wind up being less SPF than I thought, and it protects for about 2 hours! So I decided I needed to use SPF 50 under my makeup. This is the only formula that comes close to being tolerable. It’s not as lightweight as City Block, but it’s not greasy or sticky. City Block has a sheer yellow tone, which is fantastic for muting discolorations. Unfortunately Pep-Start does not have this yellow tint. It’s slightly tinted a pinky-beige color that, fortunately, doesn’t show up on my skin. I was afraid it would look ashy or just plain weird, but I don’t see any color or tint at all. There are also no white streaks, which I get from every other high SPF I’ve ever used. I can wear this on its own (without makeup over it) and it just looks like my skin. So far I haven’t broken out from it (one week’s use) but, eventually, it’s likely I’ll get some irritation. Still, most mineral sunscreens with dimethicone break me out after a couple of days, and non-mineral sunscreens after about an hour. So I’m impressed at how well my skin is tolerating this one. This works beautifully with Hydro-Blur Moisturizer, which helps to “blur“ what needs to be blurred (pores), and mattifies my makeup. I use Pep-Start UV Protector every morning. If I wear makeup, it’s my primer. If not, it’s my “BB“ cream. This has no scent that I can detect, which is another major bonus. Many sheer sunscreens smell like pure alcohol upon application, but this has NO ALCOHOL! Genius formulaion and I highly recommend this for combo-oily skin. \rIngredients:\rTitanium Dioxide 6.3%, Zinc Oxide 4%, Water, Dimethicone, Butyloctyl Salicylate, Polydiethylsiloxane, C12-15 Alkyl Benzoate, Isononyl Isononanoate, Diethylhexyl Succinate, Neopentyl Glycol Diheptanoate, Methyl Trimethicone, Butylene Glycol, Ethylhexyl Methoxycrylene, Lauryl PEG-9 Polydimethylsiloxyethyl Dimethicone, Silica, Laureth-4, Cetyl PEG/PPG 10/1 Dimethicone, Dipentaerythrityl Tri-Polyhydroxystearate, Hydrolyzed Wheat Protein/PVP Crosspolymer, Caprylyl Glycol, Dimethicone Silylate, Dimethicone /PEG-10/15 Crosspolymer, Isostearic Acid, Dimethicone Crosspolymer-3, Polyhydroxystearic Acid, Triethoxycaprylylsilane, Dipropylene Glycol, Phenoxyethanol, Iron Oxides."
735222,5,Peptide Moisturizer,"Lightweight, absorbs well, makes a difference in skin’s appearance. What more do you need?Please keep the formula & price."
515031,5,Daily UV Defense Invisible Broad Spectrum SPF 36 Sunscreen,Lightweight formula and doesn’t irritate my skin. Win win for me! I’ve repurchased a few times already.
408790,5,Vinosource-Hydra Moisturizing Sorbet,Feels nourishing and the formula absorbs quickly into my skin. I only use this at night. The cream itself is very souffle-like (it’s airy and light) and does the job! A little goes a long way.
630544,5,PLAY Everyday Sunscreen Lotion SPF 50 PA++++,"LOVE this new formula. has a very light scent to it, feels great and light on the skin, absorbs quickly. i put this on before powder and while it’s still a little wet, i put my powder on - my substitute to using a BB cream. i tried several other sunscreens - lavanila, shiseido, bobbi brown, tarte, and others BUT this one is still my favorite."


In [38]:
hydration_positive_mask = positive_reviews["review_text"].str.lower().str.contains(
    "very hydrating|so hydrating|keeps my skin hydrated|skin feels hydrated|made my skin soft|left my skin soft|moisturizing|very moisturizing",
    regex=True,
    na=False
)

positive_reviews.loc[
    hydration_positive_mask,
    ["rating", "product_name", "review_text"]
].sample(5, random_state=42)

,rating,product_name,review_text
328784,5,Luna Sleeping Retinoid Night Oil,"This stuff is honestly miraculous. I have very dry, sensitive skin and I’ve had a hard time finding a retinol product that doesn’t irritate me. I have also struggled with dermatillomania for a long time, and this works wonders on the redness and irritation that follows excessive skin-picking. The oil has an interesting botanical smell that is a little odd at first but has really grown on me, and it is incredibly calming and moisturizing. Shortly after I first got the product, I tried it out after a really bad bout of skin-picking, and I was absolutely blown away by how quickly this reduced the redness. I was skeptical that it would help minimize pore size, as I have tried many products that claim to do that but end up having no noticeable effect. To my surprise, my pores are DEFINITELY smaller. This stuff is the real deal... your skin will thank you."
305571,4,Mini Ultra Facial Moisturizing Cream with Squalane,"Love, love, love this moisturizer. My skin is hydrated and dewy looking after each use (minus of course super windy wintry days- then I would recommend something a little moisturizing)."
721537,4,SKINPOWER Airy Milky Lotion,"This is a great moisturizer that makes my skin feel baby soft. I don’t know if I should be using it for an extended amount of time to feel some long lasting changes, but I’m happy with its moisturizing qualities"
896439,5,The True Cream - Aqua Bomb Sunscreen Broad Spectrum SPF 50,"Super moisturizing, not goopy at all and spreads easily. No white cast. Unlike other products, I actually feel like I don’t need a moisturizer under this. It just feels like a moisturizer."
743432,5,Cicapair Tiger Grass Sleepair Intensive Mask,"I have very dry, sensitive skin. I really like this sleepair mask. The consistency is kind of slimy and it goes a long way. This container will last me a very long time because I don’t need to use that much product. It takes a few seconds to dry on your face and doesn’t feel weird on the skin after it has dried. When I wake up in the morning, my skin feels softer and more moisturized. I’ve also noticed a glow from using it which I absolutely love! I would definitely recommend this product if you’re looking for moisturizing radiance. I personally don’t use it every night. I like the results I get using it 2-3 times a week."


In [39]:
effective_positive_mask = positive_reviews["review_text"].str.lower().str.contains(
    "really works|works great|worked for me|saw a difference|noticed a difference|great results|made a difference|actually works",
    regex=True,
    na=False
)

positive_reviews.loc[
    effective_positive_mask,
    ["rating", "product_name", "review_text"]
].sample(5, random_state=42)

,rating,product_name,review_text
131271,5,Clear Improvement Active Charcoal Mask to Clear Pores,The only mask I’ve ever used that’s made a difference. I have super dry skin and I don’t find that this mask is too harsh as long as I follow up with a moisturizer. I use it twice a week and it really helps to clear your skin.
619638,4,Acne Solutions Clinical Clearing Gel,I received a sample of this and it surprisingly works great! It has been clearing my skin right up and I’m thinking of buying the full size. My only complaint is that it’s very drying and it smells like alcohol.
626067,5,Lotus Anti- Aging Daily Moisturizer,This product is nice to use and I noticed a difference in my skin’s hydration within a couple uses. I like to use this in the morning under my primer and it doesn’t cause me to get oilier (like some other cream moisturizers). The scent is also nice and light
294466,5,Acne Control Clarifying Cleanser,i use this with my clarisonic & i’m really pleased with it. it leaves my face feeling super clean & it works great for oil. this works best when used in conjunction with the clarifying toner & skin perfecting lotion (as part of the three step program); it cleared up my skin & has left me blemish-free. i definitely recommend this product for anyone who’s having skin troubles & is looking for something new to try.
471880,4,All About Eyes Eye Cream,"This stuff is great to add a little lovin to my undereyes, but not sure if it’s doing anything for bags, puffs, darkness etc. I haven’t noticed a difference in that area. I have noticed it’s much easier to put on undereye concealer and have it look good now that it’s so well moisturized. Also, I really like how it feels."


In [40]:
product_name = "Oat Cleansing Balm"

product_reviews = reviews[
    reviews["product_name"] == product_name
].copy()

product_reviews.shape

(3000, 18)

In [41]:
product_negative = product_reviews[
    product_reviews["rating"] <= 2
].copy()

product_positive = product_reviews[
    product_reviews["rating"] >= 4
].copy()

print("Total reviews:", len(product_reviews))
print("Negative reviews:", len(product_negative))
print("Positive reviews:", len(product_positive))

Total reviews: 3000
Negative reviews: 786
Positive reviews: 1793


In [42]:
product_complaints = count_theme_mentions(
    product_negative,
    "review_text",
    complaint_themes
)

product_praise = count_theme_mentions(
    product_positive,
    "review_text",
    praise_themes
)

product_complaints

,theme,review_count,percent_of_reviews
5,texture,34,4.33
2,breakouts,29,3.69
4,value,19,2.42
3,ineffective,16,2.04
0,dryness,4,0.51
6,smell,4,0.51
1,irritation,2,0.25
7,packaging,1,0.13


In [43]:
product_praise

,theme,review_count,percent_of_reviews
2,gentle,80,4.46
0,hydration,78,4.35
1,effective,52,2.90
5,value,25,1.39
3,texture,22,1.23
6,packaging,19,1.06
4,glow,5,0.28


In [44]:
def analyze_product(product_name, reviews, complaint_themes, praise_themes):
    product_reviews = reviews[
        reviews["product_name"] == product_name
    ].copy()

    if product_reviews.empty:
        return None

    product_negative = product_reviews[
        product_reviews["rating"] <= 2
    ].copy()

    product_positive = product_reviews[
        product_reviews["rating"] >= 4
    ].copy()

    complaints = count_theme_mentions(
        product_negative,
        "review_text",
        complaint_themes
    )

    praise = count_theme_mentions(
        product_positive,
        "review_text",
        praise_themes
    )

    summary = {
        "product_name": product_name,
        "brand_name": product_reviews["brand_name"].iloc[0],
        "price_usd": product_reviews["price_usd"].iloc[0],
        "total_reviews": len(product_reviews),
        "negative_reviews": len(product_negative),
        "positive_reviews": len(product_positive),
        "average_rating": round(product_reviews["rating"].mean(), 2)
    }

    return summary, complaints, praise

In [45]:
summary, complaints, praise = analyze_product(
    "Oat Cleansing Balm",
    reviews,
    complaint_themes,
    praise_themes
)

In [47]:
summary

{'product_name': 'Oat Cleansing Balm',
 'brand_name': 'The INKEY List',
 'price_usd': np.float64(11.99),
 'total_reviews': 3000,
 'negative_reviews': 786,
 'positive_reviews': 1793,
 'average_rating': np.float64(3.6)}

In [48]:
complaints.head(5)

,theme,review_count,percent_of_reviews
5,texture,34,4.33
2,breakouts,29,3.69
4,value,19,2.42
3,ineffective,16,2.04
0,dryness,4,0.51


In [49]:
praise.head(5)

,theme,review_count,percent_of_reviews
2,gentle,80,4.46
0,hydration,78,4.35
1,effective,52,2.90
5,value,25,1.39
3,texture,22,1.23


In [50]:
summary, complaints, praise = analyze_product(
    "Caffeine 5% + EGCG Depuffing Eye Serum",
    reviews,
    complaint_themes,
    praise_themes
)

In [51]:
summary

{'product_name': 'Caffeine 5% + EGCG Depuffing Eye Serum',
 'brand_name': 'The Ordinary',
 'price_usd': np.float64(8.9),
 'total_reviews': 2114,
 'negative_reviews': 490,
 'positive_reviews': 1405,
 'average_rating': np.float64(3.77)}

In [52]:
complaints.head(5)

,theme,review_count,percent_of_reviews
3,ineffective,79,16.12
0,dryness,16,3.27
4,value,13,2.65
1,irritation,4,0.82
2,breakouts,3,0.61


In [53]:
praise.head(5)

,theme,review_count,percent_of_reviews
1,effective,126,8.97
0,hydration,28,1.99
5,value,22,1.57
2,gentle,10,0.71
3,texture,10,0.71


In [54]:
def generate_product_insight(summary, complaints, praise):
    top_complaint = complaints.iloc[0]
    top_praise = praise.iloc[0]

    insight = (
        f"{summary['product_name']} by {summary['brand_name']} has an average "
        f"rating of {summary['average_rating']} across {summary['total_reviews']} reviews. "
        f"The most common complaint theme is {top_complaint['theme']} "
        f"({top_complaint['percent_of_reviews']}% of negative reviews), while the "
        f"most common praise theme is {top_praise['theme']} "
        f"({top_praise['percent_of_reviews']}% of positive reviews)."
    )

    return insight

In [55]:
insight = generate_product_insight(
    summary,
    complaints,
    praise
)

print(insight)

Caffeine 5% + EGCG Depuffing Eye Serum by The Ordinary has an average rating of 3.77 across 2114 reviews. The most common complaint theme is ineffective (16.12% of negative reviews), while the most common praise theme is effective (8.97% of positive reviews).


In [59]:
def calculate_opportunity_score(summary, complaints):
    # 1 = very weak rating, 0 = excellent rating
    rating_score = min(max((4.5 - summary["average_rating"]) / 1.5, 0), 1)

    # Maxes out once a product has 3,000 reviews
    review_score = min(summary["total_reviews"] / 3000, 1)

    # A complaint appearing in 20%+ of negative reviews receives max score
    top_complaint_pct = complaints.iloc[0]["percent_of_reviews"]
    complaint_score = min(top_complaint_pct / 20, 1)

    # Small affordability component
    if summary["price_usd"] <= 15:
        price_score = 1
    elif summary["price_usd"] <= 30:
        price_score = 0.75
    elif summary["price_usd"] <= 60:
        price_score = 0.5
    else:
        price_score = 0.25

    opportunity_score = (
        rating_score * 0.40 +
        review_score * 0.25 +
        complaint_score * 0.25 +
        price_score * 0.10
    )

    return round(opportunity_score * 100, 1)

In [60]:
opportunity_score = calculate_opportunity_score(
    summary,
    complaints
)

opportunity_score

np.float64(67.2)

In [61]:
print(
    f"Opportunity Score: {opportunity_score}/100"
)

Opportunity Score: 67.2/100


In [62]:
candidate_products = [
    "Caffeine 5% + EGCG Depuffing Eye Serum",
    "Oat Cleansing Balm",
    "Vitamin C Suspension 23% + HA Spheres 2%",
    "Faded Serum for Dark Spots & Discoloration",
    "C-Tango Vitamin C Eye Cream",
    "Beste No. 9 Jelly Cleanser",
    "Salicylic Acid 2% Anhydrous Solution Pore Clearing Serum"
]

In [63]:
opportunity_results = []

for product_name in candidate_products:
    result = analyze_product(
        product_name,
        reviews,
        complaint_themes,
        praise_themes
    )

    if result is None:
        continue

    summary, complaints, praise = result

    score = calculate_opportunity_score(
        summary,
        complaints
    )

    opportunity_results.append({
        "product_name": summary["product_name"],
        "brand_name": summary["brand_name"],
        "average_rating": summary["average_rating"],
        "total_reviews": summary["total_reviews"],
        "top_complaint": complaints.iloc[0]["theme"],
        "top_complaint_pct": complaints.iloc[0]["percent_of_reviews"],
        "opportunity_score": score
    })

In [64]:
opportunity_df = pd.DataFrame(opportunity_results)

opportunity_df.sort_values(
    "opportunity_score",
    ascending=False
)

,product_name,brand_name,average_rating,total_reviews,top_complaint,top_complaint_pct,opportunity_score
0,Caffeine 5% + EGCG Depuffing Eye Serum,The Ordinary,3.77,2114,ineffective,16.12,67.2
1,Oat Cleansing Balm,The INKEY List,3.60,3000,texture,4.33,64.4
5,Beste No. 9 Jelly Cleanser,Drunk Elephant,3.94,2692,breakouts,15.12,61.3
2,Vitamin C Suspension 23% + HA Spheres 2%,The Ordinary,3.45,1113,irritation,3.64,51.8
4,C-Tango Vitamin C Eye Cream,Drunk Elephant,3.75,1483,ineffective,10.85,48.4
3,Faded Serum for Dark Spots & Discoloration,Topicals,3.66,919,ineffective,9.76,47.3
6,Salicylic Acid 2% Anhydrous Solution Pore Clearing Serum,The Ordinary,3.85,330,breakouts,7.25,39.1


In [67]:
candidate_pool = products[
    (products["primary_category"] == "Skincare") &
    (products["rating"].notna()) &
    (products["rating"] < 4.1) &
    (products["reviews"] >= 300) &
    (products["loves_count"] >= 10000)
].copy()

candidate_pool.shape

(175, 16)

In [68]:
products.shape

(8494, 16)

In [69]:
candidate_pool = products[
    (products["primary_category"] == "Skincare") &
    (products["rating"].notna()) &
    (products["rating"] < 4.1) &
    (products["reviews"] >= 300) &
    (products["loves_count"] >= 10000)
].copy()

candidate_pool.shape

(175, 16)

In [75]:
candidate_pool = candidate_pool[
    ~candidate_pool["product_name"].str.startswith("Mini ")
].copy()

candidate_pool.shape

(162, 16)

In [76]:
all_opportunities = []

for product_name in candidate_pool["product_name"].unique():
    result = analyze_product(
        product_name,
        reviews,
        complaint_themes,
        praise_themes
    )

    if result is None:
        continue

    summary, complaints, praise = result

    # skip products with no usable negative-review themes
    if complaints.empty:
        continue

    score = calculate_opportunity_score(
        summary,
        complaints
    )

    all_opportunities.append({
        "product_name": summary["product_name"],
        "brand_name": summary["brand_name"],
        "price_usd": summary["price_usd"],
        "average_rating": summary["average_rating"],
        "total_reviews": summary["total_reviews"],
        "negative_reviews": summary["negative_reviews"],
        "positive_reviews": summary["positive_reviews"],
        "top_complaint": complaints.iloc[0]["theme"],
        "top_complaint_pct": complaints.iloc[0]["percent_of_reviews"],
        "top_praise": praise.iloc[0]["theme"] if not praise.empty else None,
        "top_praise_pct": praise.iloc[0]["percent_of_reviews"] if not praise.empty else None,
        "opportunity_score": score
    })

In [77]:
opportunity_ranking = pd.DataFrame(all_opportunities)

opportunity_ranking = opportunity_ranking.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

In [81]:
opportunity_ranking = pd.DataFrame(all_opportunities)

opportunity_ranking = opportunity_ranking.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

opportunity_ranking.head(20)

,product_name,brand_name,price_usd,average_rating,total_reviews,negative_reviews,positive_reviews,top_complaint,top_complaint_pct,top_praise,top_praise_pct,opportunity_score
0,Focuspot Micro Tip Patches,Dr. Jart+,20.00,3.05,577,234,264,ineffective,20.51,effective,6.82,76.0
1,Caffeine 5% + EGCG Depuffing Eye Serum,The Ordinary,8.90,3.77,2114,490,1405,ineffective,16.12,effective,8.97,67.2
2,Even Better Eyes Dark Circle Corrector,CLINIQUE,44.00,3.24,383,144,202,ineffective,22.22,hydration,7.92,66.8
3,Lippe Balm,Drunk Elephant,18.00,3.74,1898,405,1176,value,17.78,hydration,15.05,65.8
4,Oat Cleansing Balm,The INKEY List,11.99,3.60,3000,786,1793,texture,4.33,gentle,4.46,64.4
5,Beste No. 9 Jelly Cleanser,Drunk Elephant,34.00,3.94,2692,516,1902,breakouts,15.12,gentle,4.63,61.3
6,Acne-Clear Invisible Dots,Peter Thomas Roth,32.00,3.43,554,188,327,ineffective,17.55,effective,4.59,60.1
7,Vitamin C Brightening Cream,The INKEY List,10.99,3.03,514,199,224,irritation,5.03,effective,7.14,59.8
8,Protini Polypeptide Firming Refillable Moisturizer,Drunk Elephant,68.00,3.96,6045,1199,4244,breakouts,14.10,hydration,10.01,59.5
9,Oil-Absorbing Pore Treatment Strips,Peace Out,19.00,3.98,1566,321,1162,ineffective,25.55,effective,4.82,59.4


In [78]:
products[
    products["product_name"].str.contains(
        "Oat Cleansing Balm",
        case=False,
        na=False
    )
][
    [
        "product_id",
        "product_name",
        "brand_name",
        "price_usd",
        "rating",
        "reviews"
    ]
]

,product_id,product_name,brand_name,price_usd,rating,reviews
7579,P455364,Oat Cleansing Balm,The INKEY List,11.99,3.6098,2968.0
7580,P478030,Mini Oat Cleansing Balm,The INKEY List,5.99,3.6098,2968.0


In [79]:
oat_ids = products[
    products["product_name"].str.contains(
        "Oat Cleansing Balm",
        case=False,
        na=False
    )
]["product_id"].tolist()

reviews[
    reviews["product_id"].isin(oat_ids)
].groupby(
    ["product_id", "product_name"]
).size()

product_id  product_name           
P455364     Oat Cleansing Balm         3000
P478030     Mini Oat Cleansing Balm    3000
dtype: int64

In [82]:
opportunity_ranking.to_csv(
    "../data/processed/opportunity_ranking.csv",
    index=False
)

In [83]:
pd.read_csv(
    "../data/processed/opportunity_ranking.csv"
).head()

,product_name,brand_name,price_usd,average_rating,total_reviews,negative_reviews,positive_reviews,top_complaint,top_complaint_pct,top_praise,top_praise_pct,opportunity_score
0,Focuspot Micro Tip Patches,Dr. Jart+,20.00,3.05,577,234,264,ineffective,20.51,effective,6.82,76.0
1,Caffeine 5% + EGCG Depuffing Eye Serum,The Ordinary,8.90,3.77,2114,490,1405,ineffective,16.12,effective,8.97,67.2
2,Even Better Eyes Dark Circle Corrector,CLINIQUE,44.00,3.24,383,144,202,ineffective,22.22,hydration,7.92,66.8
3,Lippe Balm,Drunk Elephant,18.00,3.74,1898,405,1176,value,17.78,hydration,15.05,65.8
4,Oat Cleansing Balm,The INKEY List,11.99,3.60,3000,786,1793,texture,4.33,gentle,4.46,64.4
